In [66]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [67]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [68]:
from dtgraph.scenarios.football_data import Football

Football.load(graph)

Flushed database: Deleted 342 nodes, deleted 492 relationships, completed after 6 ms.
Load scenario: Added 63 labels, created 63 nodes, set 608 properties, created 123 relationships, completed after 2202 ms.


### Node Rules 

In [62]:
from type_checking.environment import Environment

env = Environment("../dtgraph/type_checking/env_football.json")

Rule_player_node = Rule("""
MATCH (p:Player)
GENERATE
(x = (p):Footballer {
    name             = p.name,
    born             = p.born,
    value_score      = p.market_value_m * p.goals_career,
    elite            = p.market_value_m > 100
})
""", env=env, type_strict=True)

Rule_one = Rule("""
MATCH (p:Player)
GENERATE
((_):Footballer {
    name = p.name
})
""", env=env, type_strict=True)

Rule_player_trophy_accumulate = Rule("""
MATCH (p:Player)-[:WON_TROPHY]->(t:Trophy)
GENERATE
(():Footballer {
    name    = p.name,
    trophies = [t.name]
})
""", env=env, type_strict=True)


Rule_club_has_player = Rule("""
MATCH (c:Club)<-[:PLAYS_FOR]-(p:Player)
GENERATE
((c):Team {
    name = c.name
})-[():HAS_PLAYER {
    players = [p.name]
}]->((_):Footballer {
    name = p.name
})
""", env=env, type_strict=True)

Rule_league_has_club = Rule("""
MATCH (l:League)<-[:IN_LEAGUE]-(c:Club)
GENERATE
((l):League {
    name = l.name
})-[():CONTAINS {
    clubs = [c.name]
}]->((_):Team {
    name = c.name
})
""", env=env, type_strict=True)

Rule_player_in_league = Rule("""
MATCH (p:Player)-[:PLAYS_FOR]->(c:Club)-[:IN_LEAGUE]->(l:League)
GENERATE
((p):Footballer {
    name = p.name
})-[():PLAYS_IN {
    clubs   = [c.name],
    leagues = [l.name]
}]->((l):League {
    name = l.name
})
""", env=env, type_strict=True)


### Execute Rules

In [65]:
my_transform = Transformation([Rule_player_trophy_accumulate])
# my_transform = Transformation([func_test],env=env,explain=False)
my_transform.apply_on(graph)

Index: Added 1 index, completed after 10 ms.
prop-value p.name
AST:
PropertyAccess
    ├── var: p
    └── prop: name
prop-value [t.name]
AST:
ListExpression
    └── PropertyAccess
        ├── var: t
        └── prop: name
Rule: Added 3 labels, created 1 nodes, set 43 properties, created 0 relationships, completed after 308 ms.


308

### Abort Transformation

In [64]:
my_transform.abort()

Index: Removed 1 index, completed after 11 ms.
Abort: Deleted 16 nodes, deleted 12 relationships, completed after 65 ms.
